# Phase 2 — Classical ML
## Day 9: Classification Metrics
**Date:** April 24, 2026

### What you'll learn today
- How a confusion matrix works and what each quadrant means
- Precision, Recall, and F1 — when to use which
- ROC curve and AUC score
- Gini coefficient and its relation to AUC
- Threshold tuning to shift the precision/recall trade-off


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')
print("All imports OK")


In [ ]:
# Synthetic binary classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.7, 0.3],   # slight class imbalance
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of positive class

print(f"Test set size: {len(y_test)}")
print(f"Positive class rate: {y_test.mean():.2%}")
print(f"Predicted positive rate: {y_pred.mean():.2%}")


## Confusion Matrix

The confusion matrix is a 2x2 table that shows how many predictions were right or wrong, broken down by actual class.

```
                  Predicted Negative   Predicted Positive
Actual Negative       TN                    FP
Actual Positive       FN                    TP
```

- **TP** (True Positive): correctly predicted positive
- **TN** (True Negative): correctly predicted negative
- **FP** (False Positive): predicted positive, actually negative — "false alarm"
- **FN** (False Negative): predicted negative, actually positive — "missed case"

The key insight: accuracy alone hides problems. With 90% negatives in your data, predicting all-negative gives 90% accuracy but catches zero positives.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print()
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}")
print(f"FN={fn}  TP={tp}")
print(f"Accuracy: {(tp+tn)/(tp+tn+fp+fn):.3f}")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative","Positive"])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## Precision, Recall, and F1

**Precision** = TP / (TP + FP)
> "Of all the positives I predicted, how many were actually positive?"
> Use this when false alarms are expensive. (e.g. spam filter — you don't want to block real email)

**Recall** = TP / (TP + FN)
> "Of all the actual positives, how many did I catch?"
> Use this when missing a positive is expensive. (e.g. cancer screening — you don't want to miss a case)

**F1** = 2 * (Precision * Recall) / (Precision + Recall)
> The harmonic mean of precision and recall. Good when both matter and classes are imbalanced.

There's a trade-off: pushing recall up usually pulls precision down, and vice versa.


In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print()
print("Full classification report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


## ROC Curve and AUC

A classifier outputs a probability score. You pick a **threshold** (default 0.5) to convert that to a class label. But what if you tried every possible threshold?

The **ROC curve** (Receiver Operating Characteristic) plots:
- **TPR (True Positive Rate)** = Recall = TP / (TP + FN) on the Y axis
- **FPR (False Positive Rate)** = FP / (FP + TN) on the X axis

At threshold=1.0, you predict nothing positive → TPR=0, FPR=0 (top-left corner is best).
At threshold=0.0, you predict everything positive → TPR=1, FPR=1 (bottom-right is worst).

**AUC** (Area Under the Curve) summarizes the whole curve in one number:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random guessing (the diagonal line)
- AUC < 0.5 → worse than random (flip your predictions!)

AUC is useful because it's threshold-independent — it measures the model's ability to rank positives above negatives.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc:.4f}")


## Gini Coefficient

The Gini coefficient is popular in credit scoring and lending. It's directly related to AUC:

```
Gini = 2 * AUC - 1
```

- Gini = 0 → random model
- Gini = 1 → perfect model

So if AUC = 0.85, Gini = 0.70. Banks often report Gini instead of AUC — same information, different scale.

**Why does this work?** The Gini coefficient measures how much better the model ranks positives above negatives compared to random. AUC = 0.5 (random) maps to Gini = 0, and AUC = 1.0 (perfect) maps to Gini = 1.0.


In [ ]:
gini = 2 * auc - 1
print(f"AUC:   {auc:.4f}")
print(f"Gini:  {gini:.4f}")
print()

# Demonstrate across a range
for auc_val in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    gini_val = 2 * auc_val - 1
    print(f"AUC={auc_val:.1f}  →  Gini={gini_val:.1f}")


## Threshold Tuning

The default threshold of 0.5 is rarely optimal. You can move it to favor precision or recall depending on your business goal.

**Example scenarios:**
- Fraud detection → lower threshold (catch more fraud, even if some false alarms)
- Email spam filter → higher threshold (avoid blocking real email)
- Medical diagnosis → lower threshold (missing a disease is worse than a false alarm)

The **Precision-Recall curve** shows how both metrics change as you move the threshold. You pick the point that best matches your trade-off.


In [ ]:
# Show how precision and recall change with threshold
thresholds_range = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds_range:
    y_pred_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds_range, precisions, label='Precision', color='steelblue')
plt.plot(thresholds_range, recalls, label='Recall', color='coral')
plt.plot(thresholds_range, f1s, label='F1', color='green', linestyle='--')
plt.axvline(0.5, color='gray', linestyle=':', label='Default (0.5)')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Find the threshold that maximizes F1
best_idx = np.argmax(f1s)
best_threshold = thresholds_range[best_idx]
print(f"Best threshold for F1: {best_threshold:.2f}")
print(f"At that threshold -> Precision={precisions[best_idx]:.3f}, Recall={recalls[best_idx]:.3f}, F1={f1s[best_idx]:.3f}")
print()
print(f"Default threshold (0.5) -> Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")


In [ ]:
# Precision-Recall curve
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(rec_curve, prec_curve, color='darkorange', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Tricky Bits — Common Mistakes

These are the mistakes almost everyone makes when starting out.


In [ ]:
# Mistake 1: Using accuracy with imbalanced classes
y_all_negative = np.zeros_like(y_test)  # predict everything as negative

acc = (y_all_negative == y_test).mean()
rec = recall_score(y_test, y_all_negative, zero_division=0)
print(f"All-negative model: Accuracy={acc:.2%}, Recall={rec:.2%}")
print("High accuracy, zero recall. This model is useless for finding positives!")


In [ ]:
# Mistake 2: Forgetting that F1 is the harmonic mean (not arithmetic)
p, r = 0.9, 0.1
arithmetic_mean = (p + r) / 2
harmonic_mean = 2 * p * r / (p + r)
print(f"Precision={p}, Recall={r}")
print(f"Arithmetic mean: {arithmetic_mean:.3f}  <- misleadingly high")
print(f"F1 (harmonic):   {harmonic_mean:.3f}  <- correctly penalizes the gap")
print("F1 punishes extreme imbalance between precision and recall.")


In [ ]:
# Mistake 3: Comparing AUC across very different datasets
# AUC = 0.75 on a balanced dataset is not the same as AUC = 0.75 on a 99/1 imbalanced dataset.
# Always check the Precision-Recall curve for imbalanced problems.
try:
    # This is valid - just showing the concept
    y_fake_imbalanced = np.array([1]*5 + [0]*95)
    y_fake_scores = np.random.rand(100)
    auc_imbalanced = roc_auc_score(y_fake_imbalanced, y_fake_scores)
    print(f"Random model on 95/5 imbalanced data: AUC={auc_imbalanced:.3f}")
    print("AUC can look decent even with a random model on imbalanced data.")
    print("Use Precision-Recall AUC (average_precision_score) for heavily imbalanced data.")
except Exception as e:
    print(f"Error: {e}")


## Trick Questions

Test yourself before looking at the answers!

**Q1:** A model has 99% accuracy on a dataset where 99% of samples are class 0. Is this a good model?
<details>
<summary>Answer</summary>

No. Predicting class 0 for everything gives 99% accuracy automatically. Check recall for class 1 — it will be 0%. Use F1 or AUC instead.
</details>

---

**Q2:** Your precision is 0.8 and recall is 0.8. What is your F1?
<details>
<summary>Answer</summary>

F1 = 2 * (0.8 * 0.8) / (0.8 + 0.8) = 0.8. When precision equals recall, F1 equals them too. The harmonic mean only penalizes imbalance between the two.
</details>

---

**Q3:** AUC = 0.6. Your colleague says the model is better than random. True or false?
<details>
<summary>Answer</summary>

True. AUC = 0.5 is random. AUC = 0.6 means the model ranks a random positive above a random negative 60% of the time. It's better than random, just not by a lot.
</details>

---

**Q4:** You lower the decision threshold from 0.5 to 0.3. What happens to precision and recall?
<details>
<summary>Answer</summary>

Recall goes up (you catch more positives), but precision goes down (more false positives sneak in). Lower threshold = more aggressive at calling positive = higher recall, lower precision.
</details>

---

**Q5:** A credit model has AUC = 0.82. What is its Gini coefficient?
<details>
<summary>Answer</summary>

Gini = 2 * AUC - 1 = 2 * 0.82 - 1 = 0.64
</details>


## Exercises

Fill in the `___` blanks and run each cell. The `assert` at the end will tell you if you got it right.


In [ ]:
# Exercise 1: Compute precision manually from the confusion matrix
cm_ex = confusion_matrix(y_test, y_pred)
tn_ex, fp_ex, fn_ex, tp_ex = cm_ex.ravel()

# Precision = TP / (TP + FP)
my_precision = ___ / (___ + ___)

assert abs(my_precision - precision_score(y_test, y_pred)) < 0.001, "Not quite, check the formula"
print(f"Precision: {my_precision:.4f} ✓")


In [ ]:
# Exercise 2: Compute recall manually
# Recall = TP / (TP + FN)
my_recall = ___ / (___ + ___)

assert abs(my_recall - recall_score(y_test, y_pred)) < 0.001, "Not quite, check the formula"
print(f"Recall: {my_recall:.4f} ✓")


In [ ]:
# Exercise 3: Compute F1 from precision and recall
# F1 = 2 * P * R / (P + R)
p = precision_score(y_test, y_pred)
r = recall_score(y_test, y_pred)
my_f1 = ___ * ___ * ___ / (___ + ___)

assert abs(my_f1 - f1_score(y_test, y_pred)) < 0.001, "Check the F1 formula"
print(f"F1: {my_f1:.4f} ✓")


In [ ]:
# Exercise 4: Compute Gini from AUC
auc_ex = roc_auc_score(y_test, y_prob)
my_gini = ___ * ___ - ___

assert abs(my_gini - (2 * auc_ex - 1)) < 0.001, "Gini = 2*AUC - 1"
print(f"AUC={auc_ex:.4f}, Gini={my_gini:.4f} ✓")


In [ ]:
# Exercise 5: Apply a custom threshold of 0.35 and compute F1
custom_threshold = 0.35
y_pred_custom = (y_prob >= ___).astype(int)
custom_f1 = f1_score(___, ___)

print(f"F1 at threshold 0.35: {custom_f1:.4f}")
print(f"F1 at threshold 0.50: {f1_score(y_test, y_pred):.4f}")
assert custom_f1 > 0, "Something went wrong"
print("✓")


In [ ]:
# Exercise 6: Find the number of False Negatives from the confusion matrix
cm_ex2 = confusion_matrix(y_test, y_pred)
# Hint: cm[row][col] — actual is row, predicted is col
# FN = actual positive predicted as negative
my_fn = ___

assert my_fn == confusion_matrix(y_test, y_pred).ravel()[2], "Check the confusion matrix layout"
print(f"False Negatives: {my_fn} ✓")


In [ ]:
# Exercise 7: Compute False Positive Rate (FPR = FP / (FP + TN))
tn7, fp7, fn7, tp7 = confusion_matrix(y_test, y_pred).ravel()
my_fpr = ___ / (___ + ___)

expected_fpr = fp7 / (fp7 + tn7)
assert abs(my_fpr - expected_fpr) < 0.001, "FPR = FP / (FP + TN)"
print(f"False Positive Rate: {my_fpr:.4f} ✓")


## Exercise Solutions

<details>
<summary>Click to reveal all solutions</summary>

**Exercise 1 — Precision:**
```python
my_precision = tp_ex / (tp_ex + fp_ex)
```

**Exercise 2 — Recall:**
```python
my_recall = tp_ex / (tp_ex + fn_ex)
```

**Exercise 3 — F1:**
```python
my_f1 = 2 * p * r / (p + r)
```

**Exercise 4 — Gini:**
```python
my_gini = 2 * auc_ex - 1
```

**Exercise 5 — Custom threshold:**
```python
y_pred_custom = (y_prob >= custom_threshold).astype(int)
custom_f1 = f1_score(y_test, y_pred_custom)
```

**Exercise 6 — False Negatives:**
```python
my_fn = cm_ex2[1][0]   # actual=positive (row 1), predicted=negative (col 0)
```

**Exercise 7 — FPR:**
```python
my_fpr = fp7 / (fp7 + tn7)
```

</details>


## Cumulative Review — Days 1-8

Mixed exercises covering what you've learned so far. These get you to think across topics.


In [ ]:
# Review 1 (Day 1 — Pandas): Create a DataFrame of model results and find the row with best F1
import pandas as pd
results = {
    'model': ['LogReg', 'Tree', 'RandomForest'],
    'precision': [0.78, 0.72, 0.83],
    'recall': [0.65, 0.80, 0.71],
    'f1': [0.71, 0.76, 0.77]
}
df_results = pd.DataFrame(results)

# Find the model name with the highest F1
best_model = df_results.loc[___, 'model']

assert best_model == 'RandomForest', f"Got {best_model}"
print(f"Best model by F1: {best_model} ✓")


In [ ]:
# Review 2 (Day 2 — NumPy): Compute precision from TP/FP arrays using vectorized ops
TPs = np.array([50, 30, 80])
FPs = np.array([10, 20,  5])

precisions_arr = ___ / (___ + ___)  # vectorized, no loops

expected = np.array([50/60, 30/50, 80/85])
assert np.allclose(precisions_arr, expected, atol=0.001), "Check vectorized formula"
print(f"Precisions: {precisions_arr.round(3)} ✓")


In [ ]:
# Review 3 (Day 3 — Data Cleaning): Load and clean a tiny classification results CSV
import io
raw_csv = """model,precision,recall,f1
LogReg,0.78,0.65,0.71
Tree,,0.80,0.76
RF,0.83,0.71,0.77
"""
df_clean = pd.read_csv(io.StringIO(raw_csv))

# Fill missing precision with the column mean
df_clean['precision'] = df_clean['precision'].___(___)

assert df_clean['precision'].isna().sum() == 0, "Still has nulls"
print(df_clean)
print("✓ No missing values")


In [ ]:
# Review 4 (Day 5 — PyTorch): Manually compute sigmoid and threshold predictions
import torch

logits = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
probs = torch.sigmoid(___)
preds = (probs >= 0.5).int()

expected_preds = torch.tensor([0, 0, 0, 1, 1])
assert torch.equal(preds, expected_preds), f"Got {preds}"
print(f"Probs: {probs.round(decimals=3)}")
print(f"Preds: {preds} ✓")


In [ ]:
# Review 5 (Day 6 — Pipelines): Wrap a classifier in a pipeline with standard scaling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as LR

pipe = Pipeline([
    ('scaler', ___()),
    ('clf', LR(random_state=42))
])
pipe.fit(X_train, y_train)
pipe_score = pipe.score(X_test, y_test)

assert pipe_score > 0.7, f"Score too low: {pipe_score}"
print(f"Pipeline accuracy: {pipe_score:.3f} ✓")


In [ ]:
# Review 6 (Day 7 — Decision Trees): Train a shallow tree and check feature importances
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

# Get index of most important feature
top_feature_idx = np.argmax(___)
importances = tree.feature_importances_

assert 0 <= top_feature_idx < X_train.shape[1], "Invalid feature index"
print(f"Most important feature index: {top_feature_idx}")
print(f"Importance: {importances[top_feature_idx]:.4f} ✓")


In [ ]:
# Review 7 (Day 8 — Ensembles): Compare Random Forest vs single tree by AUC
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

auc_tree = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
auc_rf   = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f"Decision Tree AUC: {auc_tree:.4f}")
print(f"Random Forest AUC: {auc_rf:.4f}")
assert auc_rf > auc_tree, "RF should beat single tree"
print("RF wins, as expected ✓")


### Cumulative Review Solutions

<details>
<summary>Click to reveal</summary>

**Review 1:**
```python
best_model = df_results['f1'].idxmax()
# then use loc[idxmax, 'model']
best_model = df_results.loc[df_results['f1'].idxmax(), 'model']
```

**Review 2:**
```python
precisions_arr = TPs / (TPs + FPs)
```

**Review 3:**
```python
df_clean['precision'] = df_clean['precision'].fillna(df_clean['precision'].mean())
```

**Review 4:**
```python
probs = torch.sigmoid(logits)
```

**Review 5:**
```python
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LR(random_state=42))])
```

**Review 6:**
```python
top_feature_idx = np.argmax(tree.feature_importances_)
```

**Review 7:**
```python
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
```

</details>


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║        DAY 9 CHEAT SHEET — Classification Metrics           ║
╠══════════════════════════════════════════════════════════════╣
║  Confusion Matrix                                            ║
║    cm = confusion_matrix(y_true, y_pred)                    ║
║    tn, fp, fn, tp = cm.ravel()                              ║
║                                                              ║
║  Core Metrics                                                ║
║    Precision  = TP / (TP + FP)  <- minimize false alarms   ║
║    Recall     = TP / (TP + FN)  <- minimize missed cases   ║
║    F1         = 2*P*R / (P+R)   <- balance both            ║
║    Accuracy   = (TP+TN) / total <- bad for imbalanced      ║
║                                                              ║
║  Sklearn                                                     ║
║    precision_score(y_true, y_pred)                          ║
║    recall_score(y_true, y_pred)                              ║
║    f1_score(y_true, y_pred)                                 ║
║    classification_report(y_true, y_pred)                   ║
║                                                              ║
║  ROC / AUC                                                   ║
║    fpr, tpr, _ = roc_curve(y_true, y_prob)                 ║
║    auc = roc_auc_score(y_true, y_prob)                      ║
║    AUC: 0.5=random, 1.0=perfect                             ║
║                                                              ║
║  Gini = 2 * AUC - 1                                         ║
║                                                              ║
║  Threshold Tuning                                            ║
║    y_pred = (y_prob >= threshold).astype(int)               ║
║    Lower threshold -> higher recall, lower precision        ║
╚══════════════════════════════════════════════════════════════╝
""")


## Next Up

**Day 10 — ShapAndProject**

You'll use SHAP to explain model predictions visually, explore feature importance plots, and build two mini projects: Customer Segmentation and Credit Risk Scoring. This is the capstone of Phase 2.
